In [14]:
from common.spark_session import spark
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import current_timestamp

In [17]:
checkpointPath = spark.sql("""
    DESCRIBE EXTERNAL LOCATION `loc_sandbox_sblakera`
""").select("url").collect()[0][0] + "checkpoints"

landingPath = spark.sql("""
    DESCRIBE EXTERNAL LOCATION `loc_sandbox_sblakera`
""").select("url").collect()[0][0] + "landing"

print(checkpointPath)
print(landingPath)

abfss://tele-sandbox@sblakera.dfs.core.windows.net/checkpoints
abfss://tele-sandbox@sblakera.dfs.core.windows.net/landing


TRAFFIC DATA

In [14]:
def readTrafficData():
    print("Reading the Raw Traffic Data :  ", end='')
    trafficSchema = StructType([
    StructField("Record_ID",IntegerType()),
    StructField("Count_point_id",IntegerType()),
    StructField("Direction_of_travel",StringType()),
    StructField("Year",IntegerType()),
    StructField("Count_date",StringType()),
    StructField("hour",IntegerType()),
    StructField("Region_id",IntegerType()),
    StructField("Region_name",StringType()),
    StructField("Local_authority_name",StringType()),
    StructField("Road_name",StringType()),
    StructField("Road_Category_ID",IntegerType()),
    StructField("Start_junction_road_name",StringType()),
    StructField("End_junction_road_name",StringType()),
    StructField("Latitude",DoubleType()),
    StructField("Longitude",DoubleType()),
    StructField("Link_length_km",DoubleType()),
    StructField("Pedal_cycles",IntegerType()),
    StructField("Two_wheeled_motor_vehicles",IntegerType()),
    StructField("Cars_and_taxis",IntegerType()),
    StructField("Buses_and_coaches",IntegerType()),
    StructField("LGV_Type",IntegerType()),
    StructField("HGV_Type",IntegerType()),
    StructField("EV_Car",IntegerType()),
    StructField("EV_Bike",IntegerType())])

    rawTrafficStream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option('cloudFiles.schemaLocation', f'{checkpointPath}/rawTrafficLoad/schemaInfer')
        .option('header', 'true')
        .schema(trafficSchema)
        .load(landingPath + '/raw_traffic/')
        .withColumn("Extract_time", current_timestamp()))
    
    rawTrafficStream.printSchema()

    return rawTrafficStream

In [23]:
def writeTrafficData(df):
    writeStream = (
        df.writeStream
            .format('delta')
            .option("checkpointLocation", checkpointPath + '/rawTrafficLoad/Checkpt')
            .outputMode('append')
            .queryName('rawTrafficWriteStream')
            .trigger(availableNow=True)
            .toTable("`sandbox-rey-01`.`bronze`.`raw_traffic`")
    )
    writeStream.awaitTermination()
    print('Write Success')

ROAD DATA

In [20]:
def readRoadData():
    print("Reading Road Data")
    roadSchema = StructType([
        StructField('Road_ID',IntegerType()),
        StructField('Road_Category_Id',IntegerType()),
        StructField('Road_Category',StringType()),
        StructField('Region_ID',IntegerType()),
        StructField('Region_Name',StringType()),
        StructField('Total_Link_Length_Km',DoubleType()),
        StructField('Total_Link_Length_Miles',DoubleType()),
        StructField('All_Motor_Vehicles',DoubleType())
        ])
    
    rawRoadStream = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option('cloudFiles.schemaLocation', f'{checkpointPath}/rawRoadLoad/schemaInfer')
            .option('header', 'true')
            .schema(roadSchema)
            .load(landingPath + '/raw_road/')
            .withColumn("Extract_time", current_timestamp())
    )

    rawRoadStream.printSchema()

    return rawRoadStream

In [ ]:
def writeRoadData(df):
    writeStream = (
        df.writeStream
            .format('delta')
            .option("checkpointLocation", checkpointPath + '/rawRoadLoad/Checkpt')
            .outputMode('append')
            .queryName('rawRoadWriteStream')
            .trigger(availableNow=True)
            .toTable("`sandbox-rey-01`.`bronze`.`raw_road`")
    )
    writeStream.awaitTermination()
    print('Write Success')

In [ ]:
readTrafficDF = readTrafficData()


Reading the Raw Traffic Data :  root
 |-- Record_ID: integer (nullable = true)
 |-- Count_point_id: integer (nullable = true)
 |-- Direction_of_travel: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Count_date: string (nullable = true)
 |-- hour: integer (nullable = true)
 |-- Region_id: integer (nullable = true)
 |-- Region_name: string (nullable = true)
 |-- Local_authority_name: string (nullable = true)
 |-- Road_name: string (nullable = true)
 |-- Road_Category_ID: integer (nullable = true)
 |-- Start_junction_road_name: string (nullable = true)
 |-- End_junction_road_name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Link_length_km: double (nullable = true)
 |-- Pedal_cycles: integer (nullable = true)
 |-- Two_wheeled_motor_vehicles: integer (nullable = true)
 |-- Cars_and_taxis: integer (nullable = true)
 |-- Buses_and_coaches: integer (nullable = true)
 |-- LGV_Type: integer (nullable = true)
 

In [21]:
readRoadDF = readRoadData()

Reading Road Data
root
 |-- Road_ID: integer (nullable = true)
 |-- Road_Category_Id: integer (nullable = true)
 |-- Road_Category: string (nullable = true)
 |-- Region_ID: integer (nullable = true)
 |-- Region_Name: string (nullable = true)
 |-- Total_Link_Length_Km: double (nullable = true)
 |-- Total_Link_Length_Miles: double (nullable = true)
 |-- All_Motor_Vehicles: double (nullable = true)
 |-- Extract_time: timestamp (nullable = false)



In [ ]:
writeTrafficData(readTrafficDF)

Write Success


In [ ]:
writeRoadData(readRoadDF)